# Problemas de contorno

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/14-problemas-de-contorno.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. distinguir um problema de contorno de um problema de valor inicial, e saber que ele pode ter nenhuma, uma ou várias soluções;
> 2. resolver problemas de contorno pelo método do tiro e por diferenças finitas, e explicar quando o tiro falha;
> 3. montar e resolver o sistema tridiagonal (linear ou, com Newton, não linear) das diferenças finitas;
> 4. usar `scipy.integrate.solve_bvp` com uma boa estimativa inicial;
> 5. tratar problemas de autovalor, como a equação de Schrödinger, como problemas de contorno.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy.integrate import solve_ivp, solve_bvp, trapezoid
from scipy.optimize import brentq
from cnlivro.edo import contorno_dif_finitas, autoestados

## 1. Problema de engenharia: o que o termopar realmente mede?

Para medir a temperatura de um gás quente que escoa num duto, o termopar fica dentro de um **poço termométrico**: um tubo de aço inoxidável fechado na ponta, que atravessa a parede do duto e protege o sensor. O termopar mede a temperatura da **ponta** do poço, e não a do gás. O calor que o gás entrega ao poço escoa, por condução, para a parede do duto, que é mais fria; e a superfície do poço troca radiação com a parede. A ponta fica, por isso, mais fria que o gás.

Ao longo do poço, com $x$ medido a partir da parede, o balanço de energia num trecho $dx$ dá

$$
k A\,\frac{d^2T}{dx^2} = h P\,(T - T_g) + \varepsilon\sigma P\,(T^4 - T_p^4),
\qquad T(0) = T_p, \qquad \frac{dT}{dx}(L) = 0,
$$ (eq-poco)

com $k$ a condutividade do aço, $A$ a área da seção do tubo, $P$ o perímetro externo, $h$ o coeficiente de convecção, $T_g$ a temperatura do gás, $T_p$ a da parede e $\varepsilon$ a emissividade. A base está presa à parede; a ponta, de área pequena, é tratada como isolada. Queremos a **leitura** $T(L)$ para vários comprimentos de imersão $L$, para escolher o poço.

As condições estão em **dois pontos diferentes**, $x = 0$ e $x = L$: é um **problema de contorno**. Não sabemos o estado completo ($T$ e $dT/dx$) em nenhuma das pontas, e não dá para simplesmente "marchar" a partir de uma delas. E o termo $T^4$ torna a equação não linear.

In [ ]:
sigma = 5.670e-8                                  # constante de Stefan-Boltzmann, W/(m² K⁴)
k_aco, h_conv, emiss = 16.0, 100.0, 0.8           # W/(m K), W/(m² K), adimensional
d_ext, d_int = 0.012, 0.006                       # m
A_sec = np.pi * (d_ext**2 - d_int**2) / 4         # m²
P_ext = np.pi * d_ext                             # m
T_gas, T_par = 500 + 273.15, 300 + 273.15         # K

def curvatura(x, T, dT, eps=emiss):
    # T'' em função de T, pela equação do poço
    return (h_conv * P_ext * (T - T_gas) + eps * sigma * P_ext * (T**4 - T_par**4)) / (k_aco * A_sec)

m_conv = np.sqrt(h_conv * P_ext / (k_aco * A_sec))
print(f"parâmetro de aleta m = √(hP/kA) = {m_conv:.1f} 1/m;  para L = 10 cm, mL = {m_conv * 0.1:.1f}")

In [ ]:
L1 = 0.10
r1 = contorno_dif_finitas(curvatura, (0, L1), T_par, ("neumann", 0.0), 400, chute=np.full(401, T_gas))
fig = go.Figure()
fig.add_trace(go.Scatter(x=r1.t * 100, y=r1.y - 273.15, mode="lines", name="temperatura do poço",
                         line=dict(color=CORES["aproximacao"], width=3)))
fig.add_trace(go.Scatter(x=[0, L1 * 100], y=[500, 500], mode="lines", name="gás, 500 °C",
                         line=dict(color=CORES["erro"], dash="dash")))
fig.add_trace(go.Scatter(x=[0, L1 * 100], y=[300, 300], mode="lines", name="parede, 300 °C",
                         line=dict(color=CORES["referencia"], dash="dash")))
fig.add_trace(go.Scatter(x=[L1 * 100], y=[r1.y[-1] - 273.15], mode="markers+text", name="leitura do termopar",
                         marker=dict(color=CORES["destaque"], size=12), text=[f"leitura: {numero(r1.y[-1] - 273.15, 1)} °C"],
                         textposition="bottom left"))
fig.update_xaxes(title="distância à parede x (cm)")
fig.update_yaxes(title="temperatura (°C)", range=[280, 520])
fig.update_layout(width=780, height=430, title="Poço de 10 cm: a ponta fica 65 K abaixo do gás",
                  legend=dict(orientation="h", y=-0.22))
mostrar(fig)

**O que notar:** a temperatura sobe depressa perto da parede e depois se estabiliza. Mas não chega à do gás: estabiliza num valor em que o calor recebido do gás por convecção se iguala ao perdido por radiação para a parede. É um erro de medida de dezenas de kelvins, que a seção 5.2 explora. Como esta curva foi calculada é o assunto do capítulo.

## 2. A ideia em uma imagem

A primeira ideia para um problema de contorno é transformá-lo num problema de valor inicial, que já sabemos resolver (capítulo 12). Na base conhecemos $T(0)$; falta a inclinação $s = T'(0)$. Chutamos $s$, integramos até a ponta e vemos se a condição lá, $T'(L) = 0$, foi satisfeita. Se não foi, corrigimos o chute. É o **método do tiro**: como um artilheiro que ajusta o ângulo do canhão pelos tiros anteriores. Arraste o slider para mudar o ângulo de tiro, num poço de 5 cm.

In [ ]:
L2 = 0.05

def sistema(x, u, eps=emiss):
    return [u[1], curvatura(x, u[0], u[1], eps)]

def disparar(s, L, eps=emiss, rtol=1e-10):
    return solve_ivp(sistema, (0, L), [T_par, s], args=(eps,), rtol=rtol, atol=1e-10, dense_output=True)

s_grade = np.linspace(6000, 11000, 120)
r_grade = np.array([disparar(s, L2).y[1, -1] for s in s_grade])
s_certo = brentq(lambda s: disparar(s, L2).y[1, -1], 8000, 9500)

def quadro_tiro(s):
    sol = disparar(s, L2)
    xx = np.linspace(0, L2, 200)
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                        subplot_titles=["o tiro: T(x) a partir da base", "a condição na ponta, T'(L), em função de s"])
    fig.add_trace(go.Scatter(x=xx * 100, y=disparar(s_certo, L2).sol(xx)[0] - 273.15, mode="lines", name="solução",
                             line=dict(color=CORES["exata"], dash="dash")), 1, 1)
    fig.add_trace(go.Scatter(x=xx * 100, y=sol.sol(xx)[0] - 273.15, mode="lines", name="tiro",
                             line=dict(color=CORES["aproximacao"], width=3)), 1, 1)
    fig.add_trace(go.Scatter(x=s_grade / 1000, y=r_grade / 1000, mode="lines", name="T'(L)",
                             line=dict(color=CORES["referencia"])), 1, 2)
    fig.add_trace(go.Scatter(x=[s / 1000], y=[sol.y[1, -1] / 1000], mode="markers", name="tiro atual",
                             marker=dict(color=CORES["aproximacao"], size=12)), 1, 2)
    fig.add_trace(go.Scatter(x=[6, 11], y=[0, 0], mode="lines", showlegend=False,
                             line=dict(color=CORES["exata"], width=1)), 1, 2)
    fig.update_xaxes(title="x (cm)", row=1, col=1)
    fig.update_yaxes(title="T (°C)", range=[50, 750], row=1, col=1)
    fig.update_xaxes(title="inclinação na base s (mil K/m)", row=1, col=2)
    fig.update_yaxes(title="T'(L) (mil K/m)", range=[-35, 45], row=1, col=2)
    fig.update_layout(width=980, height=440, showlegend=False, margin=dict(t=110),
                      title=f"s = {numero(s / 1000, 2)} mil K/m: inclinação na ponta T'(L) = "
                            f"{numero(sol.y[1, -1] / 1000, 2)} mil K/m")
    return fig

valores_s = [7000, 8000, s_certo, 9500, 10500]
quadros = [(numero(s / 1000, 2), quadro_tiro(s)) for s in valores_s]
mostrar(com_controle(quadros, rotulo="s (mil K/m) = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com inclinação pequena, a curva para de subir antes da ponta e desce; com inclinação grande, ela passa do ponto e dispara. Só um valor de $s$ chega à ponta com inclinação zero. Encontrar esse $s$ é resolver uma equação, $F(s) = T'(L; s) = 0$, cuja "função" é uma integração inteira: um problema de raízes (capítulo 3). Repare como $F$ é íngreme e curva, mesmo num poço curto.

A segunda ideia, as **diferenças finitas**, não marcha: escreve a equação em todos os pontos de uma malha **ao mesmo tempo** e resolve o sistema resultante, com as duas condições de contorno entrando como equações do sistema.

## 3. Formulação matemática

### 3.1 Problemas de contorno

Um **problema de contorno** (PC) de segunda ordem é

$$
y'' = f(x, y, y'), \qquad a < x < b,
$$

com uma condição em cada ponta: de **Dirichlet** ($y$ dado), de **Neumann** ($y'$ dado) ou de **Robin** (uma combinação). Ao contrário de um problema de valor inicial, um PC pode não ter solução, ou ter várias. Por exemplo, $y'' = -y$ com $y(0) = 0$ e $y(\pi) = c$ não tem solução se $c \ne 0$, e tem infinitas ($y = C\sin x$) se $c = 0$. No caso não linear, várias soluções são comuns (seção 7.1).

### 3.2 O método do tiro

Integramos o problema de valor inicial $y'' = f$, $y(a) = y_a$, $y'(a) = s$ e definimos o resíduo na outra ponta, por exemplo $F(s) = y(b; s) - y_b$. A solução do PC corresponde a uma raiz de $F$, que se encontra pela secante, por Newton ou por `brentq`. Se a EDO é **linear**, $y(x; s)$ é uma função afim de $s$, e dois tiros bastam: a solução é uma combinação dos dois.

O ponto fraco do tiro é o **condicionamento**. Na equação da aleta, $\theta'' = m^2\theta$, as soluções do problema de valor inicial são combinações de $e^{mx}$ e $e^{-mx}$. A solução física decai a partir da base, mas qualquer erro no chute $s$ excita o modo que **cresce**: um erro $\delta s$ vira um erro da ordem de $\delta s\,\sinh(mx)/m$ em $x$. Com $mL = 30$, $\sinh(30) \approx 5\times10^{12}$: um erro relativo de $10^{-13}$ em $s$, que é o arredondamento, vira um erro da ordem da própria solução na ponta. Em problemas não lineares é pior: o tiro errado pode **explodir** antes de chegar à outra ponta, e $F$ deixa de estar definida. O **tiro múltiplo** (dividir o intervalo e atirar em cada pedaço, casando os pedaços) reduz esse problema.

### 3.3 Diferenças finitas

Numa malha $x_i = a + i\,h$, $i = 0, \dots, n$, trocamos as derivadas por diferenças centradas (capítulo 10):

$$
\frac{y_{i+1} - 2y_i + y_{i-1}}{h^2} = f\!\left(x_i,\; y_i,\; \frac{y_{i+1} - y_{i-1}}{2h}\right), \qquad i = 1, \dots, n - 1 .
$$ (eq-dif-finitas)

São $n - 1$ equações para as incógnitas internas; $y_0$ e $y_n$ vêm das condições de Dirichlet. Uma condição de Neumann na ponta direita, $y'(b) = \beta$, usa um **nó fantasma** $y_{n+1}$, eliminado pela diferença centrada $(y_{n+1} - y_{n-1})/(2h) = \beta$; a equação {eq}`eq-dif-finitas` passa a valer também em $i = n$.

Cada equação envolve só três incógnitas vizinhas: o sistema é **tridiagonal**.

- Se $f$ é **linear** em $y$ e $y'$, é um sistema linear tridiagonal, resolvido em $O(n)$ operações (algoritmo de Thomas, capítulo 4).
- Se $f$ é **não linear**, resolve-se o sistema $\mathbf G(\mathbf y) = \mathbf 0$ pelo método de Newton (capítulo 5). A jacobiana também é tridiagonal, e cada iteração custa $O(n)$.

O erro de truncamento é $O(h^2)$, e o erro da solução também. Não há modo que cresça, porque não há marcha: as duas condições de contorno são impostas ao mesmo tempo. A função `cnlivro.edo.contorno_dif_finitas` implementa esse esquema.

### 3.4 Colocação: `solve_bvp`

`scipy.integrate.solve_bvp(fun, bc, x, y)` usa **colocação**: aproxima a solução por um polinômio cúbico em cada subintervalo, exige que a EDO seja satisfeita em pontos escolhidos de cada um, e refina a malha onde o resíduo é grande. O erro é de ordem 4. O sistema é escrito como $\mathbf y' = \mathbf f(x, \mathbf y)$ de primeira ordem, a função `bc(ya, yb)` devolve os resíduos das condições de contorno, e é preciso fornecer uma malha inicial e uma **estimativa inicial** da solução, porque o problema não linear é resolvido por Newton. Uma estimativa ruim pode levar a outra solução ou a nenhuma.

### 3.5 Problemas de autovalor

Alguns PCs têm solução não nula só para valores especiais de um parâmetro. A flambagem de uma coluna, $EI\,y'' + P\,y = 0$ com $y(0) = y(L) = 0$, só tem solução não nula para $P = (j\pi/L)^2 EI$: a menor é a **carga crítica de Euler**. A equação de Schrödinger estacionária,

$$
-\frac{\hbar^2}{2m^*}\,\psi'' + V(x)\,\psi = E\,\psi, \qquad \psi \to 0 \text{ longe do poço},
$$ (eq-schrodinger)

só tem solução para níveis de energia $E$ especiais. Com diferenças finitas, $-\psi''$ vira a matriz tridiagonal $\frac{1}{h^2}\operatorname{tridiag}(-1, 2, -1)$, e o problema vira um **problema de autovalor de matriz**, $H\boldsymbol\psi = E\boldsymbol\psi$, com $H$ simétrica: os métodos do capítulo 8 dão os níveis de energia (autovalores) e as funções de onda (autovetores) de uma vez. Pelo tiro, procura-se o $E$ para o qual a solução que parte de um lado chega ao outro decaindo.

## 4. À mão

**Diferenças finitas.** Para $y'' = y$, $y(0) = 0$, $y(1) = 1$ (solução exata $y = \sinh x/\sinh 1$), com $h = 1/3$ há duas incógnitas, $y_1 \approx y(1/3)$ e $y_2 \approx y(2/3)$:

$$
9\,(0 - 2y_1 + y_2) = y_1, \qquad 9\,(y_1 - 2y_2 + 1) = y_2
\quad\Longrightarrow\quad
\begin{cases} -19\,y_1 + 9\,y_2 = 0\\ 9\,y_1 - 19\,y_2 = -9 \end{cases}
$$

Da primeira, $y_2 = \frac{19}{9}y_1$; na segunda, $y_1\,(81 - 361)/9 = -9$, e $y_1 = 81/280 = 0{,}289286$, $y_2 = 0{,}610714$. Os valores exatos são $0{,}288921$ e $0{,}610243$: erros de $4\times10^{-4}$ com só dois pontos.

**Tiro, caso linear.** Para o mesmo problema, o tiro com inclinação $s$ dá $y(x; s) = s\sinh x$, e a condição $y(1) = 1$ dá $s = 1/\sinh 1 = 0{,}850918$. Com dois tiros numéricos quaisquer, $s_0$ e $s_1$, a secante acerta esse valor no primeiro passo, porque $F(s) = s\sinh 1 - 1$ é uma reta.

In [ ]:
r = contorno_dif_finitas(lambda x, y, dy: y, (0, 1), 0.0, ("dirichlet", 1.0), 3)
print("diferenças finitas, n = 3:", np.round(r.y, 6), "  exata:", np.round(np.sinh(r.t) / np.sinh(1), 6))

## 5. Explore

### 5.1 O leque de tiros

Para a aleta linear, $\theta'' = m^2\theta$, $\theta(0) = 1$, $\theta'(L) = 0$, a inclinação certa é $s^* = -m\tanh(mL)$, e o tiro com inclinação $s$ dá $\theta(x) = \cosh(mx) + (s/m)\sinh(mx)$, sem nenhum erro de integração. A figura mostra o tiro certo e tiros com erros **relativos** minúsculos em $s$. Arraste o slider para mudar $mL$.

In [ ]:
def potencia10(v):
    mantissa, expoente = f"{v:.0e}".split("e")
    return f"{mantissa}·10<sup>{int(expoente)}</sup>"

def quadro_leque(mL):
    xs = np.linspace(0, 1, 400)                     # x/L
    s_ex = -mL * np.tanh(mL)                        # em unidades de 1/L
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xs, y=np.cosh(mL * (1 - xs)) / np.cosh(mL), mode="lines", name="solução",
                             line=dict(color=CORES["exata"], width=3)))
    for delta, cor in [(1e-8, CORES["erro"]), (1e-12, CORES["terciaria"]), (1e-15, CORES["aproximacao"])]:
        for sinal in (1, -1):
            s = s_ex * (1 + sinal * delta)
            fig.add_trace(go.Scatter(x=xs, y=np.cosh(mL * xs) + s / mL * np.sinh(mL * xs), mode="lines",
                                     line=dict(color=cor, width=1.5), name=f"erro relativo de {delta:.0e} em s"
                                     .replace("e-0", "e-").replace("e-", "e−"), showlegend=(sinal == 1)))
    fig.update_xaxes(title="x / L")
    fig.update_yaxes(title="θ = (T − T_g)/(T_p − T_g)", range=[-0.5, 1.5])
    fig.update_layout(width=820, height=450, legend=dict(x=0.45, y=0.98),
                      title=f"mL = {mL}: um erro δ em s vira até {potencia10(np.sinh(mL))} δ na ponta")
    return fig

quadros = [(str(m_), quadro_leque(m_)) for m_ in [5, 10, 20, 30, 36]]
mostrar(com_controle(quadros, rotulo="mL = ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com $mL = 5$ ou 10, todos os tiros se confundem com a solução. Com $mL = 20$, um erro de $10^{-8}$ em $s$ já põe o tiro para fora do gráfico na ponta. Com $mL = 36$, nem o erro de $10^{-15}$, que é o arredondamento de `float64`, chega à ponta: o tiro é **impossível** em precisão dupla, por melhor que seja o método de integração ou de raízes. Diferenças finitas resolvem o mesmo problema sem dificuldade, porque impõem a condição da ponta diretamente.

### 5.2 O erro de radiação

A radiação depende da emissividade da superfície do poço: aço oxidado tem $\varepsilon \approx 0{,}8$; aço polido, cerca de $0{,}2$; um poço envolvido por uma **blindagem de radiação** (um tubo fino em volta, que fica quase na temperatura do gás) se comporta como se $\varepsilon$ fosse muito menor. Arraste o slider para mudar $\varepsilon$; cada curva é a leitura em função do comprimento de imersão, calculada por diferenças finitas.

In [ ]:
Ls = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30])

def leitura_fd(L, eps, n=300):
    r = contorno_dif_finitas(lambda x, T, dT: curvatura(x, T, dT, eps), (0, L), T_par, ("neumann", 0.0), n,
                             chute=np.full(n + 1, T_gas))
    return r.y[-1] - 273.15

In [ ]:
def quadro_radiacao(eps):
    leit = np.array([leitura_fd(L, eps) for L in Ls])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=Ls * 100, y=leit, mode="lines+markers", name="leitura do termopar",
                             line=dict(color=CORES["aproximacao"])))
    fig.add_trace(go.Scatter(x=[0, 30], y=[500, 500], mode="lines", name="temperatura do gás",
                             line=dict(color=CORES["erro"], dash="dash")))
    fig.update_xaxes(title="comprimento de imersão L (cm)")
    fig.update_yaxes(title="leitura (°C)", range=[300, 510])
    fig.update_layout(width=780, height=420, legend=dict(x=0.55, y=0.15),
                      title=f"ε = {numero(eps)}: com 30 cm, a leitura é {numero(leit[-1], 1)} °C "
                            f"(erro de {numero(500 - leit[-1], 1)} K)")
    return fig

quadros = [(numero(e), quadro_radiacao(e)) for e in [0.0, 0.05, 0.2, 0.5, 0.8]]
mostrar(com_controle(quadros, rotulo="ε = ", ativo=4, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** sem radiação ($\varepsilon = 0$), alongar o poço resolve: com uns 10 cm, a leitura chega a menos de 1 K do gás. Com radiação, a leitura **satura** abaixo do gás, e nenhum comprimento resolve: com $\varepsilon = 0{,}8$, o erro é de 65 K. O comprimento cuida da condução; contra a radiação, é preciso diminuir $\varepsilon$ (polir, blindar) ou corrigir a leitura com o modelo.

### 5.3 Um problema de autovalor: o poço quântico de um laser

Lasers semicondutores e LEDs usam **poços quânticos**: uma camada fina de GaAs, de 10 nm, entre camadas de AlGaAs, que para um elétron funcionam como um poço de potencial de profundidade 0,3 eV. Os níveis de energia permitidos no poço definem a cor da luz emitida. Com a massa efetiva do elétron no GaAs, $m^* = 0{,}067\,m_e$, o coeficiente de {eq}`eq-schrodinger` é $\hbar^2/(2m^*) = 0{,}569$ eV·nm².

Pelo **tiro**: partimos de longe, à esquerda, com $\psi = 0$ e uma inclinação pequena, e integramos para a direita. Só para energias especiais a solução volta a decair do outro lado. Arraste o slider para mudar $E$.

In [ ]:
c_q = 0.0381 / 0.067                 # ħ²/(2m*), eV·nm²
V0, meia_largura = 0.3, 5.0          # eV, nm
V_poco = lambda x: np.where(np.abs(x) < meia_largura, 0.0, V0)

# níveis exatos: equações transcendentais do poço finito (capítulo 3)
def par(E):
    kk, q = np.sqrt(E / c_q), np.sqrt((V0 - E) / c_q)
    return kk * np.sin(kk * meia_largura) - q * np.cos(kk * meia_largura)

def impar(E):
    kk, q = np.sqrt(E / c_q), np.sqrt((V0 - E) / c_q)
    return -kk * np.cos(kk * meia_largura) - q * np.sin(kk * meia_largura)

E_grade = np.linspace(1e-6, V0 - 1e-9, 20000)
niveis = []
for fn in (par, impar):
    v = fn(E_grade)
    for j in np.where(np.sign(v[:-1]) != np.sign(v[1:]))[0]:
        niveis.append(brentq(fn, E_grade[j], E_grade[j + 1]))
niveis = np.sort(niveis)
print("níveis exatos (eV):", np.round(niveis, 5))

In [ ]:
def quadro_schrodinger(E):
    s = solve_ivp(lambda x, u: [u[1], (V_poco(x) - E) / c_q * u[0]], (-15, 12), [0.0, 1e-6], rtol=1e-10,
                  atol=1e-14, max_step=0.05)
    psi = s.y[0] / np.max(np.abs(s.y[0][s.t < 6]))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=s.t, y=V_poco(s.t) / V0 * 1.5 - 0.5, mode="lines", name="potencial (fora de escala)",
                             line=dict(color=CORES["referencia"], width=1.5), fill="tozeroy",
                             fillcolor="rgba(153,153,153,0.12)"))
    fig.add_trace(go.Scatter(x=s.t, y=psi, mode="lines", name="ψ(x) pelo tiro", line=dict(color=CORES["aproximacao"], width=2.5)))
    fig.update_xaxes(title="x (nm)", range=[-15, 12])
    fig.update_yaxes(title="ψ (normalizada no poço)", range=[-2.5, 2.5])
    fig.update_layout(width=820, height=430, legend=dict(x=0.02, y=0.02),
                      title=f"E = {numero(E, 4)} eV: ψ na ponta direita = {numero(psi[-1], 1) if abs(psi[-1]) < 1e3 else f'{psi[-1]:.0e}'}")
    return fig

valores_E = [0.025, niveis[0], 0.045, 0.12, niveis[1], 0.15]
quadros = [(numero(E, 4), quadro_schrodinger(E)) for E in valores_E]
mostrar(com_controle(quadros, rotulo="E (eV) = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** para uma energia qualquer, a solução sai do poço e explode para $+\infty$ ou $-\infty$. No nível certo, ela decai dos dois lados, e o número de zeros dentro do poço conta o nível (zero para o fundamental, um para o segundo). Mesmo no nível exato, a curva acaba se afastando de zero longe do poço: é o modo crescente da seção 5.1 de novo, excitado pelo arredondamento.

Por **diferenças finitas**, todos os níveis saem de uma vez, como autovalores de uma matriz tridiagonal simétrica:

In [ ]:
E_fd, x_fd, psi_fd = autoestados(V_poco, (-20, 20), 800, k=3, coef=c_q)
for j in range(3):
    print(f"nível {j + 1}: diferenças finitas {E_fd[j]:.5f} eV, exato {niveis[j]:.5f} eV")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_fd, y=V_poco(x_fd), mode="lines", name="potencial V(x)", line=dict(color=CORES["exata"], width=2)))
cores = [CORES["aproximacao"], CORES["secundaria"], CORES["terciaria"]]
for j in range(3):
    fig.add_trace(go.Scatter(x=[-20, 20], y=[E_fd[j]] * 2, mode="lines", line=dict(color=cores[j], dash="dot", width=1),
                             showlegend=False))
    fig.add_trace(go.Scatter(x=x_fd, y=E_fd[j] + 0.06 * psi_fd[:, j] / np.abs(psi_fd[:, j]).max(), mode="lines",
                             name=f"nível {j + 1}: E = {numero(E_fd[j], 4)} eV", line=dict(color=cores[j], width=2.5)))
fig.update_xaxes(title="x (nm)", range=[-15, 15])
fig.update_yaxes(title="energia (eV)", range=[-0.03, 0.38])
fig.update_layout(width=820, height=500, title="Os três níveis ligados do poço quântico e as suas funções de onda",
                  legend=dict(orientation="h", y=-0.2))
mostrar(fig)

**O que notar:** cada função de onda está desenhada na altura da sua energia. Elas penetram um pouco nas barreiras (o efeito túnel), mais quanto mais alto o nível. Os níveis por diferenças finitas erram por menos de 1 %, com 800 subintervalos; a seção 7.2 explica por que não muito menos.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Resolva a equação do poço termométrico (com convecção e radiação) para comprimentos de imersão de 3 a 30 cm, mostre a leitura do termopar em cada caso e recomende o comprimento que dá a leitura mais próxima da temperatura do gás."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
import warnings
from scipy.integrate import solve_ivp
from scipy.optimize import newton

def rhs(x, u):
    T, dT = u
    d2T = (h_conv * P_ext * (T - T_gas) + emiss * sigma * P_ext * (T**4 - T_par**4)) / (k_aco * A_sec)
    return [dT, d2T]

def perfil_poco(L):
    # método do tiro: ajusta a inclinação na base até que dT/dx = 0 na ponta
    def residuo(s):
        sol = solve_ivp(rhs, (0, L), [T_par, s], rtol=1e-8, atol=1e-8)
        return sol.y[1, -1]
    s = newton(residuo, x0=1000.0)
    return solve_ivp(rhs, (0, L), [T_par, s], rtol=1e-8, atol=1e-8, dense_output=True)

comprimentos = [0.03, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
leituras = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for L in comprimentos:
        leituras[L] = perfil_poco(L).y[0, -1] - 273.15
        print(f"L = {L * 100:4.0f} cm: leitura = {leituras[L]:7.1f} °C")
melhor = max(leituras, key=leituras.get)
print(f"Recomendação: L = {melhor * 100:.0f} cm, com leitura de {leituras[melhor]:.1f} °C.")

O método do tiro é o que muitos livros apresentam primeiro, e o código o implementa corretamente: integra o problema de valor inicial e ajusta a inclinação com a secante (`newton` sem derivada). Não houve erro nem aviso. A recomendação, porém, contraria a intuição: um poço mais longo, mais imerso no gás, leria **pior**?

### 6.2 Verificação

**Limites físicos.** O poço recebe calor do gás e o perde para a parede, e não tem fonte interna. Pelo **princípio do máximo**, a temperatura em qualquer ponto fica entre a da parede e a do gás: $300 \le T \le 500$ °C. É um teste que não exige solução de referência:

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for L in comprimentos:
        sol = perfil_poco(L)
        T_perfil = sol.sol(np.linspace(0, L, 500))[0] - 273.15
        ok = "ok" if (T_perfil.min() >= 300 - 1e-6 and T_perfil.max() <= 500 + 1e-6) else "VIOLA"
        print(f"L = {L * 100:4.0f} cm: T mínima {T_perfil.min():8.1f} °C, T máxima {T_perfil.max():6.1f} °C  -> {ok}")

A partir de 10 cm, os perfis passam por temperaturas **abaixo do zero absoluto** ($-273{,}15$ °C). As "soluções" satisfazem $T'(L) = 0$, mas não são soluções físicas da equação.

**Contra diferenças finitas e `solve_bvp`.** Dois métodos independentes, que não marcham, com malhas diferentes:

In [ ]:
def leitura_bvp(L):
    x0 = np.linspace(0, L, 50)
    chute = np.vstack([np.full_like(x0, T_gas), np.zeros_like(x0)])
    s = solve_bvp(lambda x, u: np.vstack([u[1], curvatura(x, u[0], u[1])]),
                  lambda ua, ub: np.array([ua[0] - T_par, ub[1]]), x0, chute, tol=1e-6, max_nodes=20000)
    return s.sol(L)[0] - 273.15, s.status

print("  L     tiro (agente)   dif. finitas   solve_bvp")
for L in comprimentos:
    lb, st = leitura_bvp(L)
    print(f"{L * 100:3.0f} cm   {leituras[L]:10.2f}   {leitura_fd(L, emiss, 400):12.3f}   {lb:9.3f}  (status {st})")

Os dois métodos concordam entre si em todos os casos, e com o tiro só em 3 e 5 cm. A leitura correta **cresce** com o comprimento e satura perto de 434,9 °C.

**Balanço de energia.** O calor que o poço recebe ao longo do comprimento tem de sair pela base: $kA\,T'(0) = \int_0^L \big[hP(T_g - T) - \varepsilon\sigma P(T^4 - T_p^4)\big]dx$. A integral é calculada com a regra do trapézio (capítulo 11):

In [ ]:
L_b = 0.10
r_fd = contorno_dif_finitas(curvatura, (0, L_b), T_par, ("neumann", 0.0), 800, chute=np.full(801, T_gas))
T_b = r_fd.y
fluxo_base = k_aco * A_sec * (-3 * T_b[0] + 4 * T_b[1] - T_b[2]) / (2 * (r_fd.t[1] - r_fd.t[0]))
ganho = trapezoid(h_conv * P_ext * (T_gas - T_b) - emiss * sigma * P_ext * (T_b**4 - T_par**4), r_fd.t)
print(f"diferenças finitas, L = 10 cm: calor pela base {fluxo_base:.3f} W, calor recebido {ganho:.3f} W")

O balanço fecha. A solução por diferenças finitas está verificada por três caminhos: limites físicos, um segundo método e uma lei de conservação.

### 6.3 Por quê

O gráfico do resíduo do tiro, $F(s) = T'(L; s)$, para o poço de 10 cm, mostra o que a secante enfrentou:

In [ ]:
F10 = lambda s: solve_ivp(rhs, (0, 0.10), [T_par, s], rtol=1e-10, atol=1e-10).y[1, -1]   # resíduo do tiro: F(s) = T'(L; s)
s_raiz = brentq(F10, 8000, 9000)          # o tiro certo
s_falsa = brentq(F10, 7000, 8000)         # um zero espúrio
r10 = contorno_dif_finitas(curvatura, (0, 0.10), T_par, ("neumann", 0.0), 800, chute=np.full(801, T_gas))
s_fd = (-3 * r10.y[0] + 4 * r10.y[1] - r10.y[2]) / (2 * (r10.t[1] - r10.t[0]))
print(f"tiro certo: s = {s_raiz:.1f} K/m;  inclinação na base pelas diferenças finitas: {s_fd:.1f} K/m")
print(f"zero espúrio: s = {s_falsa:.1f} K/m, com temperatura mínima "
      f"{solve_ivp(rhs, (0, 0.10), [T_par, s_falsa], rtol=1e-10, atol=1e-10).y[0].min() - 273.15:.0f} °C")
print(f"um erro de 0,2 % em s muda F(s) em {abs(F10(s_raiz * 1.002)):.0f} K/m")

In [ ]:
s_v = np.linspace(0, 12000, 241)
F_v = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for s in s_v:
        sol = solve_ivp(rhs, (0, 0.10), [T_par, s], rtol=1e-8, atol=1e-8)
        F_v.append(sol.y[1, -1] if sol.status == 0 else np.nan)
F_v = np.array(F_v)
fig = go.Figure()
fig.add_trace(go.Scatter(x=s_v / 1000, y=np.sign(F_v) * np.log10(1 + np.abs(F_v)), mode="lines+markers",
                         marker=dict(size=4), line=dict(color=CORES["erro"]), name="F(s)"))
fig.add_vline(x=s_raiz / 1000, line=dict(color=CORES["destaque"], dash="dash"))
fig.add_annotation(x=s_raiz / 1000, y=6, text="tiro certo", showarrow=False, xanchor="left", xshift=5,
                   font=dict(color=CORES["destaque"]))
fig.add_vline(x=s_falsa / 1000, line=dict(color=CORES["referencia"], dash="dot"))
fig.add_annotation(x=s_falsa / 1000, y=-6, text="zero espúrio", showarrow=False, xanchor="right", xshift=-5,
                   font=dict(color=CORES["referencia"]))
fig.add_annotation(x=10.8, y=0, text="a integração explode<br>antes da ponta", showarrow=False,
                   font=dict(color=CORES["erro"]))
fig.update_xaxes(title="inclinação na base s (mil K/m)")
fig.update_yaxes(title="sinal(F) · log₁₀(1 + |F|)")
fig.update_layout(width=800, height=420, title="O resíduo do tiro no poço de 10 cm", showlegend=False)
mostrar(fig)

O resíduo varia por muitas ordens de grandeza (o eixo vertical é logarítmico). O tiro certo, perto de 8,8 mil K/m, coincide com a inclinação da solução por diferenças finitas, mas fica numa janela estreitíssima: acima de uns 9 mil K/m a integração nem chega à ponta, porque o termo $T^4$ faz a temperatura explodir antes. Abaixo do tiro certo, a temperatura despenca, passa por valores negativos em kelvin, e o $T^4$ volta a fazê-la subir: o resíduo cruza o zero em lugares sem significado físico, como perto de 7,7 mil K/m. Partindo de $s = 1000$, a secante encontrou um desses zeros espúrios. Quanto mais longo o poço, mais estreita a janela útil, pelo fator $e^{mL}$ da seção 3.2.

### 6.4 A correção

Diferenças finitas com Newton (ou `solve_bvp`), partindo de uma estimativa razoável (o poço todo na temperatura do gás). Para confiar no resultado, refinamos a malha e medimos a ordem:

In [ ]:
ns = np.array([25, 50, 100, 200, 400])
leit_n = np.array([leitura_fd(0.10, emiss, n) for n in ns])
dif = np.abs(np.diff(leit_n))
print("leituras para L = 10 cm:", np.round(leit_n, 5))
print("ordem observada pelas diferenças sucessivas:", np.round(np.log2(dif[:-1] / dif[1:]), 2))
leit_ok = {L: leitura_fd(L, emiss, 400) for L in comprimentos}
print("leitura correta (°C):", {f"{L * 100:.0f} cm": round(v, 1) for L, v in leit_ok.items()})

A ordem observada é 2, como prevê a seção 3.3, e a leitura com 100 subintervalos já está certa em duas casas decimais. A recomendação correta: a leitura satura em cerca de 434,9 °C a partir de uns 15 cm; aumentar o poço além disso não adianta, e o erro de 65 K vem da radiação, não do comprimento.

> **Lição.** O método do tiro herda a instabilidade do problema de valor inicial: quando a EDO tem modos que crescem ao longo do intervalo, como em aletas, poços e camadas-limite, o tiro falha **sem avisar**, e em problemas não lineares pode convergir para uma raiz espúria. Peça ao agente **diferenças finitas** ou `solve_bvp`, com uma estimativa inicial física, e verifique a solução pelos **limites físicos** (princípio do máximo), por um **balanço de energia** e pelo **refinamento da malha**. E desconfie de uma recomendação que contraria a física, como "um poço mais imerso lê pior".

## 7. Quando falha

### 7.1 Várias soluções, ou nenhuma: autoaquecimento de uma pilha

Pilhas de bagaço de cana, carvão ou grãos se aquecem sozinhas pela oxidação lenta do material. Numa pilha em forma de placa, a temperatura adimensional $\theta$ segue o modelo de Frank-Kamenetskii,

$$
\theta'' = -\delta\,e^{\theta}, \qquad \theta(-1) = \theta(1) = 0,
$$

em que $\delta$ cresce com o quadrado da espessura da pilha. Para $\delta$ pequeno há uma solução morna, estável; acima de um valor crítico $\delta_c \approx 0{,}878$ não há solução estacionária: o calor gerado não consegue sair, e a pilha entra em **combustão espontânea**. Abaixo de $\delta_c$, há ainda uma segunda solução, "quente" e instável.

In [ ]:
x7 = np.linspace(-1, 1, 201)
fk = lambda d: (lambda x, y, dy: -d * np.exp(y))
for delta in [0.5, 0.8, 0.9]:
    frio = contorno_dif_finitas(fk(delta), (-1, 1), 0.0, ("dirichlet", 0.0), 200)
    quente = contorno_dif_finitas(fk(delta), (-1, 1), 0.0, ("dirichlet", 0.0), 200, chute=6 * np.cos(np.pi * x7 / 2))
    txt_f = f"θ(0) = {frio.y[100]:.3f}" if frio.sucesso else f"falhou ({frio.motivo})"
    txt_q = f"θ(0) = {quente.y[100]:.3f}" if quente.sucesso else f"falhou ({quente.motivo})"
    print(f"δ = {delta}: chute θ = 0 -> {txt_f};  chute quente -> {txt_q}")

Com $\delta = 0{,}5$, o mesmo método, com a mesma malha, encontra soluções diferentes conforme a estimativa inicial. Com $\delta = 0{,}9$, Newton não converge de nenhum dos dois chutes, e a razão não é numérica: **não há solução**. Um agente que "resolve" a falha de convergência aumentando o número de iterações ou mudando o chute até algo convergir está escondendo a resposta de engenharia. Para encontrar $\delta_c$, a técnica certa é a **continuação**: aumentar $\delta$ aos poucos, usando cada solução como chute da seguinte, e acompanhar $\theta(0)$ até a curva dobrar.

### 7.2 Um coeficiente descontínuo derruba a ordem

No poço quântico da seção 5.3, o potencial salta de 0 para 0,3 eV na borda. A fórmula de erro das diferenças centradas supõe solução suave, e aqui $\psi''$ salta junto com $V$:

In [ ]:
def nivel_fd(n, media_na_borda=False):
    hq = 40 / n
    def V_malha(x):
        v = V_poco(x)
        if media_na_borda:   # nó exatamente sobre a borda: média dos dois lados
            v = np.where(np.isclose(np.abs(x), meia_largura, atol=hq / 10), V0 / 2, v)
        return v
    return autoestados(V_malha, (-20, 20), n, k=1, coef=c_q)[0][0]

ns = np.array([200, 400, 800, 1600])
for media in (False, True):
    e = np.array([abs(nivel_fd(n, media) - niveis[0]) for n in ns])
    print(f"{'média na borda' if media else 'valor de um lado'}: erros {', '.join(f'{v:.1e}' for v in e)}, "
          f"ordem observada {ordem_observada(40 / ns, e):.2f}")

Com o valor de um lado da descontinuidade no nó da borda, a ordem cai para 1. Usar a média dos dois lados nesse nó devolve a ordem 2. O mesmo cuidado vale para condutividades que mudam de um material para outro (uma parede composta, um poço com isolamento): coloque um nó na interface e use a média adequada, ou o erro "de ordem 2" do livro não se realiza.

### 7.3 Camada-limite numa malha grossa

Em $\varepsilon\,y'' = y'$, com $y(0) = 0$, $y(1) = 1$ e $\varepsilon$ pequeno, a solução é quase zero e sobe de repente numa camada de largura $\varepsilon$ junto a $x = 1$. Esse problema modela a transferência de calor num escoamento rápido (convecção dominando a difusão):

In [ ]:
eps_c = 0.01
exata_c = lambda x: np.expm1(x / eps_c) / np.expm1(1 / eps_c)
for n in [10, 40, 200]:
    r = contorno_dif_finitas(lambda x, y, dy: dy / eps_c, (0, 1), 0.0, ("dirichlet", 1.0), n)
    print(f"n = {n:3d}, h/ε = {1 / n / eps_c:4.1f}: menor valor {r.y.min():+.3f}, "
          f"erro máximo {np.max(np.abs(r.y - exata_c(r.t))):.2e}")

Com $h > 2\varepsilon$, a solução das diferenças centradas **oscila** e assume valores negativos, que a solução exata nunca tem. A condição $h < 2\varepsilon$ (o **número de Péclet da malha** menor que 2) garante a ausência de oscilações. Quando ela é cara demais, usam-se malhas refinadas só na camada, ou diferenças "contra o vento" (*upwind*), que o capítulo 15 apresenta.

## Resumo

| Método | Ideia | Ordem | Quando usar | Cuidado |
| --- | --- | --- | --- | --- |
| Tiro | transforma em valor inicial + raiz | a do integrador | intervalos curtos, problemas bem condicionados, autovalores 1D | falha com modos crescentes; raízes espúrias |
| Tiro múltiplo | tiros em subintervalos, casados | a do integrador | intervalos longos | mais incógnitas |
| Diferenças finitas | sistema tridiagonal (+ Newton) | 2 | padrão para PCs 1D | coeficientes descontínuos; camadas-limite |
| Colocação (`solve_bvp`) | polinômios por partes, malha adaptativa | 4 | PCs não lineares, precisão alta | precisa de boa estimativa inicial |
| Autovalores por diferenças finitas | $H\boldsymbol\psi = E\boldsymbol\psi$ tridiagonal | 2 | todos os níveis de uma vez | resolução da malha nos estados altos |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que $y'' + y = 0$, $y(0) = 0$, $y'(\pi/2) = 1$ não tem solução, e que $y(0) = 0$, $y(\pi/2) = 1$ tem uma só.
2. Escreva as equações de diferenças finitas para $y'' = y$, $y(0) = 0$, $y'(1) = 0$ com $h = 1/2$, usando o nó fantasma, e resolva à mão.
3. Para a aleta linear com $mL = 25$, estime o maior erro relativo admissível em $s$ para que o tiro erre a temperatura da ponta em menos de 1 %. É possível em `float64`?
4. Mostre que, com $V$ constante e $\psi = 0$ nas pontas de $[0, L]$, os autovalores da matriz de diferenças finitas são $\frac{2c}{h^2}\left(1 - \cos\frac{j\pi}{n}\right)$, e compare com os exatos $c\,(j\pi/L)^2$.

**Verificação (com agente)**

5. Peça a um agente a carga crítica de flambagem de uma coluna engastada na base e livre no topo, com seção variável, por diferenças finitas. Verifique com a coluna de seção constante ($P_{cr} = \pi^2EI/(4L^2)$) e meça a ordem do erro.
6. Peça a um agente para resolver o problema do poço com `solve_bvp`, partindo de uma estimativa constante igual à temperatura da **parede**. Ele converge? Para a mesma solução? Teste também o chute linear entre parede e gás.
7. Peça a um agente para encontrar $\delta_c$ do problema de Frank-Kamenetskii por continuação em $\delta$. Verifique o resultado com a solução analítica: $\theta = \theta_m - 2\ln\cosh(bx)$, com $\theta_m = 2\ln\cosh b$ e $\delta = 2b^2/\cosh^2 b$; o máximo dessa expressão em $b$ é $\delta_c \approx 0{,}878$.

**Projeto aberto**

8. **Projeto de um poço com blindagem.** Modele uma blindagem de radiação como um tubo fino em volta do poço, com a sua própria equação de aleta e troca de radiação com o poço e com a parede. Resolva o sistema de dois PCs acoplados por diferenças finitas e proponha comprimento e emissividades que deem um erro de leitura menor que 2 K.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulo 27.
- U. M. Ascher, R. M. M. Mattheij e R. D. Russell, *Numerical Solution of Boundary Value Problems for Ordinary Differential Equations*, SIAM, 1995.
- J. Kierzenka e L. F. Shampine, "A BVP solver based on residual control and the MATLAB PSE", *ACM Transactions on Mathematical Software*, 27(3), 2001 (o algoritmo do `solve_bvp`).
- F. P. Incropera e outros, *Fundamentos de Transferência de Calor e de Massa*, LTC, capítulo 3 (aletas).